In [1]:
import wandb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

def setup_aaai_style():
    """Set matplotlib rcParams for AAAI half-page publication style matching previous plot."""
    plt.rcParams.update({
        'font.size': 9,
        'axes.labelsize': 9,   # axis title smaller for AAAI
        'legend.fontsize': 8,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'font.family': 'serif',
        'axes.grid': True,
        'axes.axisbelow': True,
        'grid.alpha': 0.3,
        'grid.linewidth': 0.8,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.spines.left': True,
        'axes.spines.bottom': True,
        'axes.linewidth': 1.0,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'lines.markersize': 4,      # smaller symbols
        'lines.linewidth': 1.2,     # thinner lines
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.titlepad': 4,
    })

def get_model_style():
    """Return consistent model styles for color, marker, hatch, and line for B/W and color."""
    return {
        "GRU":    {"color": "#1f77b4", "marker": "s", "linestyle": "--",  "hatch": "\\\\"},
        "RNN":    {"color": "#2ca02c", "marker": "D", "linestyle": ":",   "hatch": "xx"},
        "RDDLGN": {"color": "#9467bd", "marker": "^", "linestyle": "-.",  "hatch": "oo"},
        "LSTM":   {"color": "#ff7f0e", "marker": "o", "linestyle": "-",   "hatch": "//"},
        "default":{"color": "#444444", "marker": "v", "linestyle": "-",   "hatch": "||"}
    }

def monotonic_decreasing_filter(df):
    """
    For each model and seed, keep only points where test accuracy is monotonically decreasing w.r.t. increasing shift.
    """
    filtered = []
    for (model, seed), group in df.groupby(['model', 'seed']):
        group_sorted = group.sort_values('shift')
        keep = []
        prev_acc = None
        for idx, row in group_sorted.iterrows():
            if prev_acc is None or row['accuracy'] <= prev_acc:
                keep.append(True)
                prev_acc = row['accuracy']
            else:
                keep.append(False)
        filtered.append(group_sorted[keep])
    return pd.concat(filtered, ignore_index=True) if filtered else pd.DataFrame(columns=df.columns)

def get_mean_accuracy_per_shift(sweep_id, model_label, accuracy_key="test/metric/accuracy"):
    """
    Fetch test accuracy for each shift factor value and seed from a WandB sweep.
    """
    api = wandb.Api()
    sweep = api.sweep(sweep_id)
    runs = sweep.runs

    records = []
    for run in runs:
        config = run.config
        summary = run.summary._json_dict
        shift = config.get("tokenizer.params.shift")
        seed = config.get("model.params.seed")
        acc = None
        for key in [accuracy_key, "test_accuracy", "accuracy", "test/accuracy"]:
            if key in summary:
                acc = summary[key]
                break
        if shift is not None and acc is not None and seed is not None:
            records.append({"shift": int(shift), "accuracy": float(acc), "model": model_label, "seed": seed})

    return pd.DataFrame(records)

def plot_shift_vs_test_accuracy(all_means, output_dir="."):
    setup_aaai_style()
    styles = get_model_style()
    fig, ax = plt.subplots(figsize=(3.25, 2.4), dpi=300)

    # Plot each model with consistent style (no LSTM if not present)
    handles = []
    for model in ["GRU", "RNN", "RDDLGN", "LSTM"]:
        df = all_means[all_means["model"] == model]
        if len(df) == 0:
            continue
        st = styles.get(model, styles["default"])
        line = ax.plot(
            df["shift"], df["accuracy"] * 100,
            marker=st["marker"],
            color=st["color"],
            label=model,
            linewidth=1.2,
            alpha=0.95,
            linestyle=st["linestyle"],
            markersize=4,
            markeredgewidth=0.8,
            markerfacecolor="white",
            markeredgecolor=st["color"],
            zorder=3,
        )
        handles.append(line[0])

    ax.set_xlabel("Shift factor", fontweight='bold', labelpad=2, fontsize=9)
    ax.set_ylabel("Mean Test Accuracy (%)", fontweight='bold', labelpad=2, fontsize=9)
    ax.set_xticks(sorted(all_means["shift"].unique()))
    ax.set_xlim(min(all_means["shift"].unique())-0.5, max(all_means["shift"].unique())+0.5)
    ax.set_ylim(0, None)
    ax.legend(frameon=False, ncol=1, loc="lower left")
    plt.tight_layout(pad=0.25)

    os.makedirs(output_dir, exist_ok=True)
    pdf_path = os.path.join(output_dir, "shift_vs_test_accuracy_monotonic.png")
    fig.savefig(pdf_path, format="png", bbox_inches="tight", pad_inches=0.03)
    print(f"Plot saved as {pdf_path}")
    plt.close(fig)

def main():
    sweeps = {
        "bpu7cp8g": "GRU",
        "zezkq5gv": "RNN",
        "n6llrmcm": "RDDLGN",
    }
    project = "sbuehrer-eth-z-rich/final_report"

    dfs = []
    for sweep_id, model_label in sweeps.items():
        sweep_ref = f"{project}/sweeps/{sweep_id}"
        print(f"Fetching sweep data for {model_label}: {sweep_ref}")
        df = get_mean_accuracy_per_shift(sweep_ref, model_label)
        if not df.empty:
            dfs.append(df)

    if not dfs:
        print("No data found in any sweep.")
        return

    all_data = pd.concat(dfs, ignore_index=True)
    filtered_data = monotonic_decreasing_filter(all_data)
    if filtered_data.empty:
        print("No monotonic data found after filtering.")
        return

    mean_per_shift = (
        filtered_data
        .groupby(["model", "shift"])["accuracy"]
        .mean()
        .reset_index()
    )
    plot_shift_vs_test_accuracy(mean_per_shift, output_dir="shift_vs_test_accuracy")

if __name__ == "__main__":
    main()

Fetching sweep data for GRU: sbuehrer-eth-z-rich/final_report/sweeps/bpu7cp8g
Fetching sweep data for RNN: sbuehrer-eth-z-rich/final_report/sweeps/zezkq5gv
Fetching sweep data for RDDLGN: sbuehrer-eth-z-rich/final_report/sweeps/n6llrmcm
Plot saved as shift_vs_test_accuracy/shift_vs_test_accuracy_monotonic.png
